# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC


# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW_CREDIT_RECORD= Path("..") / "data" / "raw" / "credit_record.csv"          # PREENCHER: nome do arquivo
RAW_APPLICATION_RECORD= Path("..") / "data" / "raw" / "application_record.csv"
PROCESSED = Path("..") / "data" / "processed" / "df_application_record_tratado.csv"
TARGET = "TARGET"                                            # PREENCHER: variável alvo

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(PROCESSED)

## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [3]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [4]:
# Definição unificada e protegida contra Data Leakage
modelos = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(class_weight="balanced", random_state=RANDOM_STATE))
    ])
}

In [5]:
# Dicionário para guardar as previsões de cada modelo
previsoes = {}

# Loop automático de treinamento e predição
for nome_modelo, pipeline in modelos.items():
    print(f"Treinando {nome_modelo}...")
    pipeline.fit(X_train, y_train)
    previsoes[nome_modelo] = pipeline.predict(X_test)
print("✅ Todos os modelos foram treinados com sucesso dentro de seus respectivos Pipelines!")


Treinando Regressão Logística...
Treinando Árvore de Decisão...
Treinando Random Forest...
Treinando Support Vector Machine...
✅ Todos os modelos foram treinados com sucesso dentro de seus respectivos Pipelines!


In [6]:
def avaliar_modelo(y_test, y_pred, nome_modelo):
    """
    Calcula e imprime as métricas de validação do scikit-learn.
    """
    print(f"Resultados para {nome_modelo}:")
    print("Acurácia:", accuracy_score(y_test, y_pred))
    print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))
    print("Relatório de Classificação:\n", classification_report(y_test, y_pred))
    print("-" * 60)


In [7]:
# Loop para avaliar todos os modelos estruturados de uma só vez
for nome_modelo, y_pred in previsoes.items():
    avaliar_modelo(y_test, y_pred, nome_modelo)


Resultados para Regressão Logística:
Acurácia: 0.5715852989577619
Matriz de Confusão:
 [[3730 2704]
 [ 420  438]]
Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.90      0.58      0.70      6434
         1.0       0.14      0.51      0.22       858

    accuracy                           0.57      7292
   macro avg       0.52      0.55      0.46      7292
weighted avg       0.81      0.57      0.65      7292

------------------------------------------------------------
Resultados para Árvore de Decisão:
Acurácia: 0.787986834887548
Matriz de Confusão:
 [[5174 1260]
 [ 286  572]]
Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.95      0.80      0.87      6434
         1.0       0.31      0.67      0.43       858

    accuracy                           0.79      7292
   macro avg       0.63      0.74      0.65      7292
weighted avg       0.87      0.79      0.82      7292

--

## 3. Validação cruzada

In [8]:
for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    print(nome, scores.mean().round(4))

Regressão Logística 0.2136
Árvore de Decisão 0.3877
Random Forest 0.4067
Support Vector Machine 0.269


In [9]:
# 1. Configuração de um validador cruzado robusto, embaralhado e estratificado
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("=" * 60)
print("   INICIANDO AVALIAÇÃO VIA VALIDAÇÃO CRUZADA (Métrica: F1)   ")
print("=" * 60)

# 2. Laço de repetição forçando o log visual de progresso
for nome, modelo in modelos.items():
    print(f"⏳ Computando os 5 blocos para: {nome}...", end="", flush=True)
    t_inicio = time.time()
    
    # Executa o cálculo usando paralelismo computacional (n_jobs=-1)
    scores = cross_val_score(
        modelo, 
        X_train, 
        y_train, 
        cv=cv_estratificado, 
        scoring="f1", 
        n_jobs=-1
    )
    
    t_fim = time.time()
    tempo_gasto = round(t_fim - t_inicio, 1)
    
    # Print com formatação limpa e centralizada para o console
    print(f"\r✅ {nome:<25} | Média F1-Score: {scores.mean().round(4):.4f} | Tempo: {tempo_gasto}s")

print("=" * 60)
print("🏁 Processo concluído com sucesso!")


   INICIANDO AVALIAÇÃO VIA VALIDAÇÃO CRUZADA (Métrica: F1)   
✅ Regressão Logística       | Média F1-Score: 0.2099 | Tempo: 1.7s
✅ Árvore de Decisão         | Média F1-Score: 0.3961 | Tempo: 1.2s
✅ Random Forest             | Média F1-Score: 0.4174 | Tempo: 3.4s
✅ Support Vector Machine    | Média F1-Score: 0.2705 | Tempo: 93.6s
🏁 Processo concluído com sucesso!


## 4. Comparação e Escolha do Modelo Vencedor

### Média do F1-Score via Validação Cruzada (5-Fold CV)
Após submeter os quatro pipelines candidatos à validação cruzada robusta com a métrica F1-Score (focada na classe minoritária de risco `1.0`), os resultados obtidos foram.


* **Regressão Logística:** Média F1-Score: 0.2099  
* **Árvore de Decisão:** Média F1-Score: 0.3961   
* **Random Forest:** Média F1-Score: 0.4174
* **Support Vector Machine (SVM):** Média F1-Score: 0.2705 

### Análise Crítica dos Resultados
1. **O Classificador Campeão:** O modelo que apresentou o melhor desempenho geral para o problema de risco de crédito foi o **Random Forest**, alcançando uma média de F1-Score de **Média F1-Score: 0.3961**. 
2. **Equilíbrio de Negócio (Precisão vs. Recall):** Modelos lineares e baseados em distância (Regressão Logística e SVM) apresentaram uma taxa de captura (Recall) satisfatória no split simples, porém sacrificaram significativamente a precisão, gerando um volume excessivo de falsos alarmes (bons pagadores barrados). Os modelos baseados em conjuntos de árvores (especialmente o Random Forest) demonstraram maior capacidade de mapear os padrões não-lineares do cadastro, equilibrando a minimização do risco de inadimplência com a experiência do cliente legítimo.
3. **Conclusão:** O **Random Forest** demonstrou ser o modelo com o melhor resultado sob validação cruzada, sendo escolhido como o classificador final para implantação no pipeline de concessão de crédito.